# Procesamiento unificado: Bios + KOF

Este notebook procesa dos fuentes:

- `data/Bios_WebScrapping.xlsx`
- `data/cbg_turnover_v23upload.xlsx`

Y deja una tabla final unificada con estas columnas:

- `name`
- `iso3`
- `country`
- `role`
- `source`


## 1. Imports y configuración

In [ ]:
import re
from pathlib import Path

import openpyxl
import pandas as pd

BIOS_FILE = Path("data/Bios_WebScrapping.xlsx")
KOF_FILE = Path("data/cbg_turnover_v23upload.xlsx")
KOF_SHEET = "governors v2023"
OUTPUT_FILE = Path("data/combined_people_roles.csv")

## 2. Funciones auxiliares

Aquí dejamos funciones para limpiar nombres, parsear años y convertir la base KOF a formato tabular.

In [ ]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text or "").strip())


def extract_year(s):
    if not s:
        return ""
    s = str(s).lower().strip()
    if any(x in s for x in ["present", "current", "ongoing"]):
        return "En cargo"
    m = re.search(r"\d{4}", s)
    return m.group() if m else ""


def clean_name(name):
    name = re.sub(
        r"^(dr\.|mr\.|mrs\.|ms\.|prof\.|ing\.|sir\s)\s*",
        "",
        str(name),
        flags=re.IGNORECASE,
    )
    name = re.sub(
        r"\s*\((?:reapp|first time|interim|acting|second|1st|2nd|3rd|reappointed)[^)]*\)",
        "",
        name,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", name).strip().rstrip(".,;")


def parse_kof_cell(text, country_name, iso):
    if not text or str(text).strip() in ["None", ""]:
        return []

    text = str(text).strip()
    skip_keywords = [
        "established", "est. in", "est.in", "code:", "from 1985 onwards member",
        "member of beac", "(beac)", "(bceao)", "(eccb)", "no central bank",
        "code: -999", "exists since", "est. 1", "est.1",
    ]
    if any(p in text.lower() for p in skip_keywords):
        return []

    patterns = [
        r"(\d{1,2}-\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+to\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+-\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+-\s+(present|current)\s+(.+)",
        r"(.+?)\s+(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})$",
        r"(.+?)\s+(\d{4})\s+to\s+(\d{4})$",
    ]

    rows = []
    for entry in re.split(r"\n", text):
        entry = entry.strip()
        if not entry:
            continue

        for i, pat in enumerate(patterns):
            m = re.search(pat, entry, re.IGNORECASE)
            if not m:
                continue

            if i < 7:
                start_raw, end_raw, name = m.group(1), m.group(2), m.group(3)
            else:
                name, start_raw, end_raw = m.group(1), m.group(2), m.group(3)

            name = clean_name(name)
            start_year = extract_year(start_raw)
            end_year = extract_year(end_raw)
            if not name or not start_year:
                continue

            rows.append(
                {
                    "name": name,
                    "iso3": iso,
                    "country": country_name,
                    "role": "President / Governor",
                    "source": "KOF",
                }
            )
            break
    return rows

## 3. Procesar Bios_WebScrapping.xlsx

De la base Bios tomamos el nombre, el ISO3 y el cargo (`Position`).

In [ ]:
bios_df = pd.read_excel(BIOS_FILE)
bios_work = bios_df[["PName_original", "PName", "iso3", "Position"]].copy()

for col in ["PName_original", "PName", "iso3", "Position"]:
    bios_work[col] = bios_work[col].fillna("").astype(str).map(normalize_spaces)

bios_work["name"] = bios_work["PName_original"].where(
    bios_work["PName_original"] != "",
    bios_work["PName"]
)
bios_work = bios_work[bios_work["name"] != ""].copy()
bios_work = bios_work.rename(columns={"Position": "role"})
bios_work["source"] = "Bios"
bios_work = bios_work[["name", "iso3", "role", "source"]]

print(bios_work.shape)
display(bios_work.head(20))

## 4. Procesar KOF desde el Excel original

Aquí reconstruimos una tabla desde la hoja `governors v2023`, donde cada columna representa un país.

In [ ]:
wb = openpyxl.load_workbook(KOF_FILE, read_only=True, data_only=True)
ws = wb[KOF_SHEET]
rows = list(ws.iter_rows(values_only=True))

iso_codes = rows[0]
country_names = rows[1]

countries = {}
for i, (iso, name) in enumerate(zip(iso_codes, country_names)):
    if iso and name:
        countries[i] = {"iso": str(iso).strip(), "name": str(name).strip()}

kof_records = []
for row in rows[2:]:
    for col_idx, val in enumerate(row):
        if col_idx not in countries:
            continue
        if not val or str(val).strip() in ["None", ""]:
            continue
        ci = countries[col_idx]
        kof_records.extend(parse_kof_cell(val, ci["name"], ci["iso"]))

kof_df = pd.DataFrame(kof_records)
print(kof_df.shape)
display(kof_df.head(20))

## 5. Construir mapa `iso3 -> country`

Usamos KOF como fuente principal para mapear países. Luego lo aplicamos a Bios.

In [ ]:
country_map = (
    kof_df[["iso3", "country"]]
    .dropna()
    .drop_duplicates(subset=["iso3"])
)

bios_with_country = bios_work.merge(country_map, on="iso3", how="left")
display(bios_with_country.head(20))

## 6. Unificar ambas fuentes

Aquí apilamos Bios y KOF en una sola tabla homogénea.

In [ ]:
combined = pd.concat(
    [
        bios_with_country[["name", "iso3", "country", "role", "source"]],
        kof_df[["name", "iso3", "country", "role", "source"]],
    ],
    ignore_index=True,
)

for col in ["name", "iso3", "country", "role", "source"]:
    combined[col] = combined[col].fillna("").astype(str).map(normalize_spaces)

combined = combined[combined["name"] != ""].copy()
combined = combined.drop_duplicates().sort_values(["iso3", "country", "name", "role", "source"])

print(combined.shape)
display(combined.head(30))

## 7. Exportar CSV final

In [ ]:
combined.to_csv(OUTPUT_FILE, index=False)
print(f"CSV guardado en: {OUTPUT_FILE}")

## 8. Ver el archivo final

In [ ]:
saved_df = pd.read_csv(OUTPUT_FILE)
print(saved_df.shape)
display(saved_df.head(30))